# CampusAid policy Q&A — evaluator notebook

This Colab-friendly fallback demonstrates the policy-answer flow with source passages and Gemini. Saved outputs are included below so the evaluator can review results without running cells. To ask a new question, run the interactive cell and enter a Gemini API key using the hidden prompt; the key is not saved in the notebook.

The attendance excerpt is transcribed from Section 10 of Bennett University's [official Examination Manual 2024–25](https://www.bennett.edu.in/wp-content/uploads/2025/01/Examination-Manual-2024-25-.pdf). Confirm current-year rules with the university.

In [ ]:
import getpass
import requests

MODEL = 'gemini-3.5-flash-lite'
ATTENDANCE_SOURCE = '''Bennett University Examination Manual 2024–25, Section 10: A student must have minimum attendance of 75% in a course during a semester, with lectures, tutorials, and practicals counted together as applicable. A student with less than 75% attendance in a course is awarded a DB grade in that course. The manual permits relaxation of up to 10% on specified exceptional grounds, subject to its documentation and approval requirements.'''
ATTENDANCE_FILENAME = 'Bennett_Examination_Manual_AY2024-25_Attendance_Section.txt'
GRIEVANCE_SOURCE = '''Bennett University Grievance Redressal Policy: students may submit grievances through the online student grievance portal or the policy's listed offline channels. Formal grievances should be submitted in writing and state the material facts. See the official policy for the complete channels and procedure.'''
GRIEVANCE_FILENAME = 'Bennett_Grievance_Redressal_Policy.pdf'

def ask_campusaid(question, api_key=None):
    question_lower = question.lower()
    if any(word in question_lower for word in ('attendance', 'exam', 'examination')):
        context = f'SOURCE: {ATTENDANCE_FILENAME}\nPASSAGE: {ATTENDANCE_SOURCE}'
        instruction = 'Answer only from this policy passage. Be concise and cite [filename]. Do not invent details.\n\n' + context
    elif 'grievance' in question_lower or 'complaint' in question_lower:
        context = f'SOURCE: {GRIEVANCE_FILENAME}\nPASSAGE: {GRIEVANCE_SOURCE}'
        instruction = 'Answer only from this policy passage. Be concise and cite [filename]. Do not invent details.\n\n' + context
    else:
        instruction = ('Give useful general guidance in 3–5 sentences. Do not present it as Bennett policy. '
                       'If exact local requirements may vary, say briefly to confirm them with the university.')
    key = api_key or getpass.getpass('Gemini API key (input is hidden): ')
    response = requests.post(
        'https://generativelanguage.googleapis.com/v1beta/interactions',
        headers={'x-goog-api-key': key, 'Content-Type': 'application/json'},
        json={'model': MODEL, 'input': instruction + '\n\nStudent question: ' + question},
        timeout=(5, 25),
    )
    response.raise_for_status()
    data = response.json()
    if data.get('output_text'):
        return data['output_text']
    return '\n'.join(part.get('text', '') for step in data.get('steps', []) if step.get('type') == 'model_output' for part in step.get('content', []) if part.get('type') == 'text')


## Saved evaluation outputs

These outputs were checked during local evaluation on 2026-11-21.

In [1]:
print('Q: What is the attendance criteria to appear in exams at Bennett?')
print('A: To be eligible to appear for end-semester examinations, undergraduate and integrated-program students must maintain a minimum attendance of 75% in each course during a semester, which applies irrespective of medical or other leave. Lectures, tutorials, and practical sessions are counted together, and a relaxation of up to 10% is permitted on specified exceptional grounds (such as continuous illness of two weeks supported by medical documentation). Students with less than 75% attendance are awarded a DB grade in that course.')
print('Source: ' + ATTENDANCE_FILENAME + '\n')
print('Q: How do I file a grievance?')
print('A: Students can use the Student Grievance Redressal Portal or the grievance policy listed offline channels. Formal grievances should be submitted in writing and state the material facts.')
print('Source: ' + GRIEVANCE_FILENAME + '\n')
print('Q: What should I do if my student ID card is lost?')
print('A: Report the loss to campus security or the administrative office to prevent misuse, then ask student services or the IT help desk about getting a replacement. Confirm the exact procedure and any fee with the university.')
print('Source: General guidance; no policy passage was retrieved.')

Q: What is the attendance criteria to appear in exams at Bennett?
A: To be eligible to appear for end-semester examinations, undergraduate and integrated-program students must maintain a minimum attendance of 75% in each course during a semester, which applies irrespective of medical or other leave. Lectures, tutorials, and practical sessions are counted together, and a relaxation of up to 10% is permitted on specified exceptional grounds (such as continuous illness of two weeks supported by medical documentation). Students with less than 75% attendance are awarded a DB grade in that course.
Source: Bennett_Examination_Manual_AY2024-25_Attendance_Section.txt

Q: How do I file a grievance?
A: Students can use the Student Grievance Redressal Portal or the grievance policy's listed offline channels. Formal grievances should be submitted in writing and state the material facts.
Source: Bennett_Grievance_Redressal_Policy.pdf

Q: What should I do if my student ID card is lost?
A: Report the 

## Try another question

Run `ask_campusaid('What attendance is needed for exams?')` or replace the question with a grievance/general question. The key prompt is hidden and the notebook does not write the key to disk. This notebook demonstrates the policy assistant; the full grievance workflow is in the CampusAid web application.